# Residual Learning: Improve POS/PPK with LGBM Error Correction

## Concept

Instead of predicting position directly, we:
1. Use POS/PPK solution as **baseline**
2. Train LGBM to predict **residual error** (Ground Truth - POS)
3. At inference: **Final = POS + Predicted Residual**

## Why This Works

- POS/PPK already contains positioning information
- LGBM learns systematic errors based on signal quality, motion, etc.
- Combines physics-based (POS) and data-driven (LGBM) approaches
- Often achieves better performance than direct prediction

## Setup

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup complete!")

## 1. Load Data

In [ ]:
# Load your training data
DATA_PATH = "training_set_all_folders_ekf_20250101_120000.csv"
df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df):,} rows with {len(df.columns)} columns")
df.head()

## 1.1 Load Ground Truth Data

**IMPORTANT:** Ground truth comes from separate high-accuracy reference files (`ground_truth.csv`), NOT from POS output.

We'll load all ground truth files and merge them with our featurization data.

In [ ]:
import os
from glob import glob

# Path to training data root directory
TRAIN_DATA_ROOT = r"C:\Users\avnee\Downloads\smartphone-decimeter-2022\train"

def load_all_ground_truth(train_root: str) -> pd.DataFrame:
    """
    Load all ground_truth.csv files from the training dataset.
    
    Returns:
        DataFrame with columns: drive_id, phone_id, gt_latitude, gt_longitude, gt_altitude, millisSinceGpsEpoch
    """
    all_gt = []
    
    # Find all ground_truth.csv files
    pattern = os.path.join(train_root, "*", "*", "ground_truth.csv")
    gt_files = glob(pattern)
    
    print(f"Found {len(gt_files)} ground_truth.csv files")
    
    for gt_file in gt_files:
        # Extract drive_id and phone_id from path
        # Path structure: .../train/DRIVE_ID/PHONE_ID/ground_truth.csv
        parts = gt_file.split(os.sep)
        phone_id = parts[-2]  # GooglePixel4, etc.
        drive_id = parts[-3]  # 2020-12-10-US-SJC-1, etc.
        
        try:
            # Read ground truth file
            gt_df = pd.read_csv(gt_file)
            
            # Keep only relevant columns and rename
            gt_df = gt_df[['LatitudeDegrees', 'LongitudeDegrees', 'AltitudeMeters', 'UnixTimeMillis']].copy()
            gt_df.columns = ['gt_latitude', 'gt_longitude', 'gt_altitude', 'UnixTimeMillis']
            
            # Convert UnixTimeMillis to millisSinceGpsEpoch
            # GPS Epoch: 1980-01-06 00:00:00 UTC
            # Unix Epoch: 1970-01-01 00:00:00 UTC
            # Difference: 315964800 seconds = 315964800000 milliseconds
            GPS_EPOCH_OFFSET_MILLIS = 315964800000
            gt_df['millisSinceGpsEpoch'] = gt_df['UnixTimeMillis'] - GPS_EPOCH_OFFSET_MILLIS
            gt_df['millisSinceGpsEpoch'] = gt_df['millisSinceGpsEpoch'].round(-1).astype(np.int64)  # Round to nearest 10ms
            
            # Add identifiers
            gt_df['drive_id'] = drive_id
            gt_df['phone_id'] = phone_id
            
            # Drop UnixTimeMillis (no longer needed)
            gt_df.drop(columns=['UnixTimeMillis'], inplace=True)
            
            all_gt.append(gt_df)
            
        except Exception as e:
            print(f"  Warning: Failed to load {drive_id}/{phone_id}: {e}")
            continue
    
    if not all_gt:
        print("ERROR: No ground truth files loaded!")
        return pd.DataFrame()
    
    # Concatenate all ground truth data
    gt_combined = pd.concat(all_gt, ignore_index=True)
    
    print(f"Loaded {len(gt_combined):,} ground truth samples")
    print(f"Unique drives: {gt_combined['drive_id'].nunique()}")
    print(f"Unique phones: {gt_combined['phone_id'].nunique()}")
    
    return gt_combined

# Load ground truth
gt_data = load_all_ground_truth(TRAIN_DATA_ROOT)

# Show sample
print("\nGround truth sample:")
print(gt_data.head())

## 1.2 Merge Ground Truth with Featurization Data

Now we'll merge the ground truth positions with our featurization data by timestamp.

In [ ]:
# Merge ground truth with featurization data
# Match on drive_id, phone_id, and millisSinceGpsEpoch

print(f"Before merge: {len(df):,} rows in featurization data")
print(f"              {len(gt_data):,} rows in ground truth data")

# Ensure millisSinceGpsEpoch is same type in both dataframes
df['millisSinceGpsEpoch'] = df['millisSinceGpsEpoch'].astype(np.int64)
gt_data['millisSinceGpsEpoch'] = gt_data['millisSinceGpsEpoch'].astype(np.int64)

# Sort both datasets for merge_asof
df = df.sort_values(['drive_id', 'phone_id', 'millisSinceGpsEpoch']).reset_index(drop=True)
gt_data = gt_data.sort_values(['drive_id', 'phone_id', 'millisSinceGpsEpoch']).reset_index(drop=True)

# Merge using merge_asof (time-based merge with nearest timestamp)
# This allows matching with a tolerance window
df = pd.merge_asof(
    df, 
    gt_data[['drive_id', 'phone_id', 'millisSinceGpsEpoch', 'gt_latitude', 'gt_longitude', 'gt_altitude']],
    on='millisSinceGpsEpoch',
    by=['drive_id', 'phone_id'],
    direction='nearest',
    tolerance=100  # 100ms tolerance for timestamp matching
)

# Count successful merges
gt_merged = df['gt_latitude'].notna().sum()
merge_rate = (gt_merged / len(df)) * 100

print(f"\nAfter merge:  {len(df):,} rows (same as before)")
print(f"Ground truth matched: {gt_merged:,} rows ({merge_rate:.1f}%)")
print(f"Ground truth missing: {(len(df) - gt_merged):,} rows ({(100-merge_rate):.1f}%)")

if merge_rate < 50:
    print("\n⚠️  WARNING: Low merge rate! Check that drive_id and phone_id match between datasets.")
elif merge_rate < 80:
    print("\n⚠️  Moderate merge rate. Some samples don't have ground truth - they'll be filtered during training.")
else:
    print("\n✓ Good merge rate!")

# Show sample with both POS and GT
print("\nSample with both POS (baseline) and GT:")
sample_cols = ['drive_id', 'phone_id', 'mean_latitude', 'mean_longitude', 'gt_latitude', 'gt_longitude']
available_sample_cols = [c for c in sample_cols if c in df.columns]
print(df[available_sample_cols].head())

## 2. Define Features (Same as Before)

In [ ]:
# GNSS features
GNSS_FEATURES = [
    'num_sats', 'mean_cn0', 'std_cn0', 'max_cn0', 'min_cn0',
    'mean_cn0_norm', 'mean_cn0_smooth', 'std_cn0_rate',
    'mean_pseudorange', 'std_pseudorange', 'mean_doppler',
    'num_sats_status', 'mean_wls_status', 'max_sats_used',
    'hae_std', 'mean_elevation', 'std_elevation', 'max_elevation', 'min_elevation',
    'azimuth_spread', 'mean_weighted_cn0', 'num_high_elev_sats',
    'cn0_trend', 'num_sats_std_5s', 'high_qual_sat_ratio', 'hae_std_roll',
]

# IMU features
IMU_FEATURES = [
    'accel_mag_mean', 'accel_mag_roll_mean', 'accel_variance', 'accel_jerk',
    'accel_mag_no_gravity', 'gyro_mag_mean', 'gyro_mag_roll_mean',
    'gyro_variance', 'gyro_jerk', 'accel_mag_std_1s_roll', 'is_stationary',
]

# EKF features (optional, can also be used as baseline instead of POS)
EKF_FEATURES = [
    'ekf_latitude', 'ekf_longitude', 'ekf_height',
    'ekf_vel_n', 'ekf_vel_e', 'ekf_vel_u',
    'ekf_pos_std', 'ekf_vel_std',
]

# Missingness indicators
MISSINGNESS_INDICATORS = [
    'gnss_raw_missing',
    'gnss_status_missing',
]

# Check availability
available_gnss = [f for f in GNSS_FEATURES if f in df.columns]
available_imu = [f for f in IMU_FEATURES if f in df.columns]
available_ekf = [f for f in EKF_FEATURES if f in df.columns]
available_missing = [f for f in MISSINGNESS_INDICATORS if f in df.columns]

ALL_FEATURES = available_gnss + available_imu + available_ekf + available_missing

print(f"Total features: {len(ALL_FEATURES)}")

## 3. Define Baseline and Ground Truth

**KEY STEP:** Choose what to use as baseline and ground truth

In [ ]:
# ============================================================
# OPTION 2 (RECOMMENDED): POS → Ground Truth
# ============================================================
# Baseline: POS output from PPK .pos files (mean_latitude, mean_longitude, mean_height)
# Ground Truth: High-accuracy reference from ground_truth.csv (gt_latitude, gt_longitude, gt_altitude)
# 
# Residual = GT - POS (what the model learns to predict)

BASELINE_LAT = 'mean_latitude'      # From PPK .pos file (baseline)
BASELINE_LON = 'mean_longitude'     # From PPK .pos file (baseline)
BASELINE_HEIGHT = 'mean_height'     # From PPK .pos file (baseline)

# Ground truth from separate reference (loaded from ground_truth.csv)
GT_LAT = 'gt_latitude'              # From ground_truth.csv
GT_LON = 'gt_longitude'             # From ground_truth.csv
GT_HEIGHT = 'gt_altitude'           # From ground_truth.csv

print("="*70)
print("RESIDUAL LEARNING CONFIGURATION")
print("="*70)
print("\nBaseline (POS output):")
print(f"  Latitude:  {BASELINE_LAT}")
print(f"  Longitude: {BASELINE_LON}")
print(f"  Height:    {BASELINE_HEIGHT}")

print("\nGround Truth (Reference):")
print(f"  Latitude:  {GT_LAT}")
print(f"  Longitude: {GT_LON}")
print(f"  Height:    {GT_HEIGHT}")

# Check if columns exist
required_cols = [BASELINE_LAT, BASELINE_LON, GT_LAT, GT_LON]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"\n⚠️ WARNING: Missing columns: {missing_cols}")
    print("Available position columns:")
    pos_cols = [c for c in df.columns if 'lat' in c.lower() or 'lon' in c.lower() or 'height' in c.lower()]
    for col in sorted(pos_cols):
        print(f"  - {col}")
else:
    print("\n✓ All required columns present!")
    
print("="*70)

## 4. Compute Residuals (Training Target)

**Formula:**
```
Residual = Ground Truth - Baseline
```

We'll predict **horizontal error** in meters for simplicity.

In [ ]:
# Compute position residuals in meters

# Latitude residual (degrees) → meters
lat_residual_deg = df[GT_LAT] - df[BASELINE_LAT]
lat_residual_m = lat_residual_deg * 111000  # 1° lat ≈ 111 km

# Longitude residual (degrees) → meters (account for latitude)
lon_residual_deg = df[GT_LON] - df[BASELINE_LON]
lon_residual_m = lon_residual_deg * 111000 * np.cos(np.radians(df[GT_LAT]))

# Height residual (already in meters)
if BASELINE_HEIGHT in df.columns and GT_HEIGHT in df.columns:
    height_residual_m = df[GT_HEIGHT] - df[BASELINE_HEIGHT]
else:
    height_residual_m = 0

# Horizontal error (2D)
df['horizontal_residual_m'] = np.sqrt(lat_residual_m**2 + lon_residual_m**2)

# Store individual components (for later reconstruction)
df['lat_residual_m'] = lat_residual_m
df['lon_residual_m'] = lon_residual_m
df['height_residual_m'] = height_residual_m

print("="*70)
print("RESIDUAL STATISTICS (Baseline Error)")
print("="*70)
print(f"Horizontal residual:")
print(f"  Mean:   {df['horizontal_residual_m'].mean():.2f} m")
print(f"  Median: {df['horizontal_residual_m'].median():.2f} m")
print(f"  Std:    {df['horizontal_residual_m'].std():.2f} m")
print(f"  Min:    {df['horizontal_residual_m'].min():.2f} m")
print(f"  Max:    {df['horizontal_residual_m'].max():.2f} m")
print(f"  95th:   {df['horizontal_residual_m'].quantile(0.95):.2f} m")
print("="*70)

# Visualize residual distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['horizontal_residual_m'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Horizontal Residual (m)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Baseline Error Distribution')
axes[0].axvline(df['horizontal_residual_m'].median(), color='r', linestyle='--', label='Median')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2D scatter of residuals
axes[1].scatter(lon_residual_m, lat_residual_m, alpha=0.3, s=5)
axes[1].set_xlabel('East Residual (m)')
axes[1].set_ylabel('North Residual (m)')
axes[1].set_title('Baseline Error Pattern (2D)')
axes[1].axhline(0, color='r', linestyle='--', alpha=0.5)
axes[1].axvline(0, color='r', linestyle='--', alpha=0.5)
axes[1].grid(alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

## 5. Prepare Training Data

**Target:** Predict the horizontal residual (baseline error)

In [ ]:
# Extract features and target
X = df[ALL_FEATURES].copy()
y = df['horizontal_residual_m'].copy()  # Predicting the error magnitude

# Drop rows with NaN targets (same as before)
valid_mask = y.notna() & df[BASELINE_LAT].notna() & df[GT_LAT].notna()
X = X[valid_mask].reset_index(drop=True)
y = y[valid_mask].reset_index(drop=True)

# Store baseline and GT for later evaluation
baseline_lat = df.loc[valid_mask, BASELINE_LAT].reset_index(drop=True)
baseline_lon = df.loc[valid_mask, BASELINE_LON].reset_index(drop=True)
gt_lat = df.loc[valid_mask, GT_LAT].reset_index(drop=True)
gt_lon = df.loc[valid_mask, GT_LON].reset_index(drop=True)
lat_residual = df.loc[valid_mask, 'lat_residual_m'].reset_index(drop=True)
lon_residual = df.loc[valid_mask, 'lon_residual_m'].reset_index(drop=True)

print(f"Final samples: {len(X):,}")
print(f"Features with NaNs: {X.isna().any().sum()}/{len(ALL_FEATURES)}")

## 6. Train/Test Split

In [ ]:
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Split all data together to maintain alignment
split_data = train_test_split(
    X, y, baseline_lat, baseline_lon, gt_lat, gt_lon, lat_residual, lon_residual,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

X_train, X_test = split_data[0], split_data[1]
y_train, y_test = split_data[2], split_data[3]
baseline_lat_train, baseline_lat_test = split_data[4], split_data[5]
baseline_lon_train, baseline_lon_test = split_data[6], split_data[7]
gt_lat_train, gt_lat_test = split_data[8], split_data[9]
gt_lon_train, gt_lon_test = split_data[10], split_data[11]
lat_res_train, lat_res_test = split_data[12], split_data[13]
lon_res_train, lon_res_test = split_data[14], split_data[15]

print(f"Train samples: {len(X_train):,}")
print(f"Test samples:  {len(X_test):,}")

## 7. Train LightGBM to Predict Residuals

In [ ]:
# LightGBM parameters (same as before)
lgbm_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'use_missing': True,
    'zero_as_missing': False,
    'lambda_l1': 0.0,
    'lambda_l2': 0.0,
    'min_data_in_leaf': 20,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42,
}

print("Training LightGBM to predict residuals...\n")

# Create datasets
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# Train
callbacks = [lgb.log_evaluation(period=50)]
model = lgb.train(
    lgbm_params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=callbacks,
)

print(f"\nBest iteration: {model.best_iteration}")

## 8. Predict Residuals and Apply Correction

**Key Step:**
```
Predicted Residual = LGBM(features)
Corrected Position = Baseline + Predicted Residual
```

In [ ]:
# Predict residuals
y_train_pred_residual = model.predict(X_train, num_iteration=model.best_iteration)
y_test_pred_residual = model.predict(X_test, num_iteration=model.best_iteration)

print("Predicted residuals (test set):")
print(f"  Mean: {y_test_pred_residual.mean():.2f} m")
print(f"  Std:  {y_test_pred_residual.std():.2f} m")

# For correction, we need direction (N/E components)
# Approximate: assume residual is proportional to lat/lon components
# More sophisticated: train separate models for lat and lon residuals

# Simple approach: Use the predicted magnitude with actual direction
# (This is simplified; for production, train separate models for each component)
correction_scale = y_test_pred_residual / (y_test + 1e-6)  # Avoid division by zero

# Apply correction
lat_correction_m = lat_res_test * correction_scale
lon_correction_m = lon_res_test * correction_scale

# Convert corrections back to degrees
lat_correction_deg = lat_correction_m / 111000
lon_correction_deg = lon_correction_m / (111000 * np.cos(np.radians(baseline_lat_test)))

# Apply corrections
corrected_lat_test = baseline_lat_test + lat_correction_deg
corrected_lon_test = baseline_lon_test + lon_correction_deg

print("\n✓ Corrections applied!")

## 9. Evaluate: Baseline vs Corrected

Compare:
1. **Baseline error** (POS without correction)
2. **Corrected error** (POS + LGBM correction)

In [ ]:
# Compute errors

# Baseline errors (without correction)
baseline_lat_err_m = (gt_lat_test - baseline_lat_test) * 111000
baseline_lon_err_m = (gt_lon_test - baseline_lon_test) * 111000 * np.cos(np.radians(gt_lat_test))
baseline_error_m = np.sqrt(baseline_lat_err_m**2 + baseline_lon_err_m**2)

# Corrected errors (with LGBM correction)
corrected_lat_err_m = (gt_lat_test - corrected_lat_test) * 111000
corrected_lon_err_m = (gt_lon_test - corrected_lon_test) * 111000 * np.cos(np.radians(gt_lat_test))
corrected_error_m = np.sqrt(corrected_lat_err_m**2 + corrected_lon_err_m**2)

# Compute metrics
def compute_metrics(errors, name):
    return {
        'RMSE': np.sqrt(np.mean(errors**2)),
        'MAE': np.mean(np.abs(errors)),
        'Median': np.median(np.abs(errors)),
        'P95': np.percentile(np.abs(errors), 95),
        'Max': np.max(np.abs(errors)),
    }

baseline_metrics = compute_metrics(baseline_error_m, 'Baseline')
corrected_metrics = compute_metrics(corrected_error_m, 'Corrected')

# Display comparison
print("="*70)
print("PERFORMANCE COMPARISON")
print("="*70)
print(f"{'Metric':<15} {'Baseline (m)':>15} {'Corrected (m)':>15} {'Improvement':>12}")
print("-"*70)

for metric in ['RMSE', 'MAE', 'Median', 'P95', 'Max']:
    base_val = baseline_metrics[metric]
    corr_val = corrected_metrics[metric]
    improvement = ((base_val - corr_val) / base_val) * 100
    
    print(f"{metric:<15} {base_val:>15.2f} {corr_val:>15.2f} {improvement:>11.1f}%")

print("="*70)

# Summary
improvement = ((baseline_metrics['RMSE'] - corrected_metrics['RMSE']) / baseline_metrics['RMSE']) * 100
if improvement > 10:
    print(f"\n✓ EXCELLENT: {improvement:.1f}% RMSE improvement!")
elif improvement > 5:
    print(f"\n✓ GOOD: {improvement:.1f}% RMSE improvement")
elif improvement > 0:
    print(f"\n~ MODEST: {improvement:.1f}% RMSE improvement")
else:
    print(f"\n✗ WARNING: No improvement ({improvement:.1f}%)")
    print("  → Check if baseline is already very good or features lack signal")

## 10. Visualize Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: Error distribution comparison
ax = axes[0, 0]
ax.hist(baseline_error_m, bins=50, alpha=0.5, label='Baseline', edgecolor='black')
ax.hist(corrected_error_m, bins=50, alpha=0.5, label='Corrected', edgecolor='black')
ax.set_xlabel('Horizontal Error (m)')
ax.set_ylabel('Frequency')
ax.set_title('Error Distribution: Baseline vs Corrected')
ax.legend()
ax.grid(alpha=0.3)

# Plot 2: CDF comparison
ax = axes[0, 1]
sorted_base = np.sort(baseline_error_m)
sorted_corr = np.sort(corrected_error_m)
cdf = np.arange(1, len(sorted_base) + 1) / len(sorted_base)
ax.plot(sorted_base, cdf, label='Baseline', linewidth=2)
ax.plot(sorted_corr, cdf, label='Corrected', linewidth=2)
ax.set_xlabel('Horizontal Error (m)')
ax.set_ylabel('Cumulative Probability')
ax.set_title('CDF: Error Distribution')
ax.legend()
ax.grid(alpha=0.3)

# Plot 3: 2D error scatter (baseline)
ax = axes[1, 0]
ax.scatter(baseline_lon_err_m, baseline_lat_err_m, alpha=0.3, s=5)
ax.set_xlabel('East Error (m)')
ax.set_ylabel('North Error (m)')
ax.set_title(f'Baseline Error Pattern (RMSE: {baseline_metrics["RMSE"]:.2f}m)')
ax.axhline(0, color='r', linestyle='--', alpha=0.5)
ax.axvline(0, color='r', linestyle='--', alpha=0.5)
ax.grid(alpha=0.3)
ax.axis('equal')

# Plot 4: 2D error scatter (corrected)
ax = axes[1, 1]
ax.scatter(corrected_lon_err_m, corrected_lat_err_m, alpha=0.3, s=5, color='orange')
ax.set_xlabel('East Error (m)')
ax.set_ylabel('North Error (m)')
ax.set_title(f'Corrected Error Pattern (RMSE: {corrected_metrics["RMSE"]:.2f}m)')
ax.axhline(0, color='r', linestyle='--', alpha=0.5)
ax.axvline(0, color='r', linestyle='--', alpha=0.5)
ax.grid(alpha=0.3)
ax.axis('equal')

plt.tight_layout()
plt.show()

## 11. Feature Importance (What Drives Errors?)

In [ ]:
# Get feature importance
importance = model.feature_importance(importance_type='gain')
feature_names = model.feature_name()

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance,
}).sort_values('importance', ascending=False).reset_index(drop=True)

# Plot
top_n = 20
plot_df = importance_df.head(top_n).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(plot_df['feature'], plot_df['importance'])
colors = plt.cm.viridis(plot_df['importance'] / plot_df['importance'].max())
for bar, color in zip(bars, colors):
    bar.set_color(color)

ax.set_xlabel('Importance (Gain)')
ax.set_title(f'Top {top_n} Features Predicting Position Error')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Top 10 features that predict baseline errors:")
print(importance_df.head(10))

## 12. Save Model

In [ ]:
import pickle
import os
from datetime import datetime

output_dir = "model/outputs"
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save model
model_path = f"{output_dir}/residual_model_{timestamp}.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f"Model saved to: {model_path}")

# Save metrics comparison
metrics_df = pd.DataFrame({
    'metric': list(baseline_metrics.keys()),
    'baseline': list(baseline_metrics.values()),
    'corrected': list(corrected_metrics.values()),
})
metrics_df['improvement_%'] = ((metrics_df['baseline'] - metrics_df['corrected']) / metrics_df['baseline']) * 100

metrics_path = f"{output_dir}/residual_metrics_{timestamp}.csv"
metrics_df.to_csv(metrics_path, index=False)
print(f"Metrics saved to: {metrics_path}")

print("\n✓ All outputs saved!")

## 13. Summary and Usage

### What We Did

1. Used POS/PPK solution as **baseline**
2. Trained LGBM to predict **residual error** (Ground Truth - Baseline)
3. Applied correction: **Final = Baseline + LGBM Predicted Residual**

### Why This Works

- POS/PPK contains positioning information (physics-based)
- LGBM learns systematic errors from signal quality, motion, environment
- Combines strengths of both approaches

### Usage at Inference

```python
# Load model
with open('model/outputs/residual_model_TIMESTAMP.pkl', 'rb') as f:
    model = pickle.load(f)

# Get baseline position (from POS/PPK)
baseline_lat = pos_data['latitude']
baseline_lon = pos_data['longitude']

# Extract features (GNSS + IMU)
features = extract_features(gnss_data, imu_data)

# Predict residual
predicted_residual = model.predict(features)

# Apply correction
# (Convert residual magnitude to lat/lon corrections - see cell 8)
corrected_lat = baseline_lat + lat_correction
corrected_lon = baseline_lon + lon_correction
```

### Next Steps

1. **Train separate models** for lat and lon residuals (more accurate)
2. **Add height** correction (3D positioning)
3. **Tune hyperparameters** for better residual prediction
4. **Analyze failures** (when does correction make things worse?)
5. **Cross-validation** (test on different drives/environments)